**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**02 | MATLAB Object Decoding and Numerical Signal Recovery**

![Original MATLAB decoding and method-specific scientific validation](../../images/02_consolidated_workflow.svg)

**Execution:** The simplified cells have not been rerun on the full dataset. Numerical findings described here refer to the previously completed run. Each notebook reads the current upstream files; re-execution replaces its own output files. No historical reports, hash comparisons or PASSED gates are required.


**00. Colab Repository Setup**


In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

/content
/content/human-movement-intelligence


**01. Imports and Environment**

In [ ]:
# ======================== 01. Imports and Environment ========================

from python_modules.core.colab_setup import setup_colab

root = setup_colab("biomechanical_load_estimation")
import sys
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
from scipy.io import whosmat
from scipy.io.matlab import matfile_version
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans",
    title_size=12.5,
    title_weight="normal",
    label_size=11.5,
    label_weight="normal",
    tick_size=11,
    legend_size=10,
    figure_dpi=100,
    figure_size=(9, 4),
)
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ARTICLE_ID,
    ARTICLE_VERSION,
    ensure_matlab_archive,
    inspect_matlab_archive,
    extract_matlab_files,
)
from essex_numeric_recovery import (
    select_diagnostic_records,
    inspect_sampled_records,
    candidate_inventory,
    activity_fe_coverage,
)
from essex_python_first import (
    PILOT_SUBJECTS,
    select_representative_rows,
    probe_pilot,
    evaluate_pilot,
    inspect_native_table_schema,
)
from essex_record_audit import summarise_trial_evidence

study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
output_dir = root / "biomechanical_load_estimation" / "output"
raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print("Original Figshare version:", ARTICLE_ID, ARTICLE_VERSION)
print("Research outputs saved directly to:", output_dir)

**02. Current Dataset and Trial Registry**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 02. Current Dataset and Trial Registry

zip_path = ensure_matlab_archive(raw_dir)
inventory = inspect_matlab_archive(zip_path)
mat_paths = extract_matlab_files(zip_path, mat_dir, inventory)
registry = pd.read_csv(
    output_dir / "01_original_record_registry.csv",
    dtype={
        "subject_id": str,
        "source_file": str,
        "local_record_key": str,
        "task": str,
        "updated_trial_id": str,
        "fe_representation": str,
        "imu_representation": str,
    },
)
source_summary = summarise_trial_evidence(registry)
display(
    pd.DataFrame(
        [
            {
                "participants": registry["subject_id"].nunique(),
                "trials": len(registry),
                "activities": registry["task"].nunique(),
                "empty_fe_records": int(
                    registry["fe_representation"].eq("empty").sum()
                ),
            }
        ]
    )
)

**03. Source FE Coverage and MATLAB Containers**

Report task-level and participant-level source FE missingness, inspect MATLAB container variables and record file-level structural evidence. A source-empty field and a nonnumeric object representation must remain distinct.


In [ ]:
# =============== 03. Source FE Coverage and MATLAB Containers ================

activity_counts = (
    registry.groupby("task", dropna=False)
    .size()
    .rename("record_count")
    .reset_index()
    .sort_values("record_count", ascending=False)
)
fe_coverage = activity_fe_coverage(registry)
fe_table = (
    fe_coverage.pivot(index="task", columns="fe_representation", values="record_count")
    .fillna(0)
    .astype(int)
)
display(activity_counts)
display(fe_table)
subject_coverage = (
    registry.groupby(["subject_id", "fe_representation"])
    .size()
    .rename("record_count")
    .reset_index()
)
fe_coverage.to_csv(output_dir / "02_bounded_activity_fe_coverage.csv", index=False)
subject_coverage.to_csv(
    output_dir / "02_bounded_participant_fe_coverage.csv", index=False
)
print("Original activities:", registry["task"].nunique())
print(
    "FE fields missing by representation:",
    int((registry["fe_representation"] == "empty").sum()),
)
container_rows = []
for source in mat_paths:
    major, minor = matfile_version(source)
    for variable_name, shape, matlab_class in whosmat(source):
        container_rows.append(
            {
                "source_file": source.name,
                "mat_major": int(major),
                "mat_minor": int(minor),
                "source_variable": variable_name,
                "source_shape": str(tuple(shape)),
                "matlab_class": matlab_class,
            }
        )
containers = pd.DataFrame(container_rows)
display(containers.head(12))
containers.to_csv(output_dir / "02_bounded_matlab_container_inventory.csv", index=False)
print("MAT container variables recorded:", len(containers))
print(
    "Files with MATLAB workspace reference:",
    containers.loc[
        containers["source_variable"] == "__function_workspace__", "source_file"
    ].nunique(),
)

**04. Bounded SciPy Object and Numeric Leaf Audit**

Use a reproducible, bounded inspection of original MATLAB objects to preserve nested paths, numeric leaf candidates and the limits of shape-based inference. Do not treat candidate leaves as prediction-ready FE values.


In [ ]:
# ============== 04. Bounded SciPy Object and Numeric Leaf Audit ==============

selected_records = select_diagnostic_records(registry, max_samples=64)
display(
    selected_records[
        [
            "subject_id",
            "task",
            "updated_trial_id",
            "local_record_key",
            "fe_representation",
        ]
    ].head(16)
)
print("Source records selected for bounded introspection:", len(selected_records))
payload_paths, probe_summary = inspect_sampled_records(
    mat_dir, selected_records, max_depth=9, max_nodes=300
)
payload_paths.to_csv(
    output_dir / "02_bounded_matlab_object_path_inventory.csv", index=False
)
probe_summary.to_csv(output_dir / "02_bounded_probe_registry.csv", index=False)
display(
    probe_summary.groupby("modality")[
        ["nodes_inspected", "numeric_leaves_visible", "inspection_truncated"]
    ].sum()
)
print(
    "Truncated source-object probes:", int(probe_summary["inspection_truncated"].sum())
)
candidate_table = candidate_inventory(payload_paths)
candidate_table.to_csv(
    output_dir / "02_bounded_numeric_leaf_candidates.csv", index=False
)
print("Numeric leaves discovered:", len(candidate_table))
print(
    "Shape-compatible with 101 or 11 positions:",
    int(candidate_table["time_axis_candidate"].sum()) if len(candidate_table) else 0,
)
if candidate_table.empty:
    print("No attributable numerical IMU/FE arrays were recovered in sampled objects.")
else:
    display(
        candidate_table[
            [
                "subject_id",
                "modality",
                "path",
                "shape",
                "dtype",
                "beneath_matlab_object",
                "time_axis_candidate",
                "usable_for_prediction",
            ]
        ].head(25)
    )
time_candidates = candidate_table.loc[candidate_table["time_axis_candidate"]].copy()
time_candidates.to_csv(
    output_dir / "02_bounded_temporal_axis_candidates.csv", index=False
)
print("Original MATLAB sensor / target column names: NOT YET VALIDATED")
print("Prediction-ready numeric tensor constructed: NO")

**05. Pinned Python Table Decoder**

Install the specifically tested original MATLAB table reader and verify its version. The source identities and output labels must remain unchanged through decoding.


In [ ]:
# ====================== 05. Pinned Python Table Decoder ======================

%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"
from matio import load_from_mat

print("Python reader:", version("mat-io"))
print("Python runtime:", sys.version.split()[0])

**06. Source-Linked Pilot Selection and Container Preflight**

Select the authentic four-participant pilot from approved original trial IDs; inspect each source MAT container, retaining full file and trial traceability. Use a bounded sample instead of repeatedly decoding the complete archive.


In [ ]:
# ========= 06. Source-Linked Pilot Selection and Container Preflight =========

selected = select_representative_rows(registry, subjects=PILOT_SUBJECTS, max_each=8)
selected.to_csv(output_dir / "02_python_pilot_selection.csv", index=False)
display(
    selected[
        [
            "subject_id",
            "task",
            "updated_trial_id",
            "local_record_key",
            "fe_representation",
        ]
    ]
)
print("Original pilot records:", len(selected))
preflight_rows = []
for subject in PILOT_SUBJECTS:
    path = mat_dir / f"{subject}.mat"
    for variable, shape, matlab_class in whosmat(path):
        preflight_rows.append(
            {
                "subject_id": subject,
                "source_file": path.name,
                "variable": variable,
                "shape": str(tuple(shape)),
                "matlab_class": matlab_class,
            }
        )
preflight = pd.DataFrame(preflight_rows)
preflight.to_csv(output_dir / "02_python_source_preflight.csv", index=False)
display(preflight)

**07. Python IMU and FE Decoding with Column Schema**

Decode original labelled IMU and FE MATLAB objects from the pilot, record reader errors, and retain original column names and time-index metadata. Descriptive column types are not in themselves proof of validated numerical target matrices.


In [ ]:
# ============= 07. Python IMU and FE Decoding with Column Schema =============

pilot, errors, decoded = probe_pilot(load_from_mat, mat_dir, selected)
pilot.to_csv(output_dir / "02_python_decode_audit.csv", index=False)
errors.to_csv(output_dir / "02_python_reader_errors.csv", index=False)
print("Original record-modality probes:", len(pilot))
print("File-level decoding errors:", len(errors))
print("Original-column-labelled numeric candidates:", len(decoded))
display(
    pilot[
        [
            "source_file",
            "local_record_index",
            "modality",
            "decode_status",
            "source_identity_match",
            "python_class",
            "shape",
            "n_time",
            "n_channels",
        ]
    ].head(30)
)
if not errors.empty:
    display(errors)
column_schema, index_schema = inspect_native_table_schema(
    load_from_mat, mat_dir, selected
)
column_schema.to_csv(output_dir / "02_python_column_schema.csv", index=False)
index_schema.to_csv(output_dir / "02_python_index_schema.csv", index=False)
column_summary = (
    column_schema.groupby(
        [
            "modality",
            "original_column_name",
            "pandas_dtype",
            "is_numeric_dtype",
            "is_time_label_candidate",
        ],
        dropna=False,
    )
    .size()
    .rename("source_records")
    .reset_index()
)
print("Original table-column metadata rows:", len(column_schema))
print("Source-index metadata records:", len(index_schema))
display(column_summary.head(100))
display(
    index_schema.groupby(
        ["modality", "table_class", "n_rows", "n_columns", "index_class"], dropna=False
    )
    .size()
    .rename("source_records")
    .reset_index()
)
print("No original column excluded or relabelled.")

**08. Python Decoder Results**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 08. Python Decoder Results

pilot_results = evaluate_pilot(pilot, errors)
status = (
    pilot.groupby(["modality", "decode_status"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)
status.to_csv(output_dir / "02_python_decoding_status.csv", index=False)
candidates = pilot.loc[pilot["decode_status"].eq("labelled_numeric_table")].copy()
candidates.to_csv(output_dir / "02_python_labelled_candidates.csv", index=False)
display(status)
display(pd.DataFrame([pilot_results]))
print("The full-cohort numerical recovery follows in Notebook 03.")

**09. Original Decoder Evidence and Real IMU Plot**

Plot authentic pilot reader classifications and a real decoded IMU column, without generating unsupported FE curves or repeating the source-level distribution figure from Notebook 01.


In [ ]:
# ============== 09. Original Decoder Evidence and Real IMU Plot ==============

fig, ax = plt.subplots(
    1,
    2,
    figsize=(13.5, 5),
    gridspec_kw={"width_ratios": [1.65, 1.0]},
    constrained_layout=True,
)
by_status = (
    status.pivot(index="decode_status", columns="modality", values="count")
    .fillna(0)
    .astype(int)
    .sort_index()
)
x = np.arange(len(by_status))
w = 0.34
for off, label in [(-0.5, "IMU"), (0.5, "FE")]:
    values = (
        by_status[label].to_numpy() if label in by_status else np.zeros(len(by_status))
    )
    ax[0].bar(x + off * w, values, w, label=label)
ax[0].set_xticks(x)
ax[0].set_xticklabels(
    [v.replace("_", " ") for v in by_status.index], rotation=25, ha="right"
)
ax[0].set_title("Original Python MATLAB object decoding outcomes")
ax[0].set_ylabel("Original pilot record-modality entries")
ax[0].legend(frameon=False)
ax[0].grid(axis="y", alpha=0.2)
imu_keys = [k for k in decoded if k[1] == "IMU"]
if imu_keys:
    real_values, original_names = decoded[imu_keys[0]]
    ax[1].plot(np.arange(real_values.shape[0]), real_values[:, 0], linewidth=1.5)
    ax[1].set_title("Original numeric IMU column (units unverified)")
    ax[1].set_xlabel("Source sample index")
    ax[1].set_ylabel("Numerical value")
else:
    ax[1].text(
        0.5,
        0.5,
        "No source-labelled numerical IMU waveform",
        ha="center",
        va="center",
        transform=ax[1].transAxes,
    )
    ax[1].set_title("Authentic numerical waveform availability")
    ax[1].set_xticks([])
    ax[1].set_yticks([])
save_figure(fig, output_dir / "02_python_decoder_diagnostics.png")
plt.show()

**10. Results Summary and Output Download**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 10. Results Summary and Output Download

display(
    pd.DataFrame(
        [
            {
                "source_trials": len(registry),
                "pilot_trials": len(selected),
                "modality_observations": len(pilot),
                "reader_error_files": len(errors),
                "labelled_imu_tables": pilot_results["labelled_numeric_imu"],
                "labelled_fe_tables": pilot_results["labelled_numeric_fe"],
            }
        ]
    )
)
print("Notebook 03 reads the current Python column schema and original trial registry.")
download_outputs_from_colab(folder="biomechanical_load_estimation/output", stage="02")

**11. Scientific Interpretation and Next Research Boundary**

**Limitations:** No fabricated FE, no interpolation to 101 points, no physical unit claims, no unbiased participant-level Train/Test protocol, and no external or clinical validation. A consistency pass is not a model-readiness claim.